# Paneles solares: mejoras de Faster R-CNN

Variantes de `entrenamiento_colab.ipynb`. Se prueban tres cambios, uno encima del otro; todo lo demás (folds, congelamiento, optimizador, métricas) es igual:

- **Anclas `paneles`**: tamaños de 16 a 256 px y proporciones alto/ancho 0.25, 0.5, 1, 2 y 4. Con las de torchvision (32 a 512 px, proporciones 0.5 a 2), el 27 % de las cajas no tiene ningún ancla con IoU >= 0.5; con estas, el 8 %.
- **20 épocas sin early stopping**: el coseno llega a 0 en la última época. En las corridas base cortaban con el learning rate todavía entre el 30 % y el 80 % del máximo.
- **Aumento `fuerte`**: además de los volteos, rotación de 0/90/180/270 grados y cambios de brillo, contraste, saturación y tono. Apunta a lo que muestran las curvas de la base: la ResNet50 ajusta el train pero no generaliza a la fuente nueva.

Se corre solo la mejor configuración de la base, **ResNet50 `partial`** (mAP50-95 macro 0.331), en los 3 folds. `full` empataba con ella (0.325, dentro del ruido entre épocas), así que correrla duplicaría el tiempo sin aportar mucho.

### Variantes

Cada variante agrega un cambio sobre la anterior. La ablación (`coco + 20 épocas`) es el control que separaría el efecto de las anclas del de las épocas; queda disponible pero apagada:

| Variante | Anclas | Épocas | Carpeta |
|---|---|---|---|
| base (ya corrida) | coco | 50 con early stopping | `RCNN` |
| ablación (opcional, apagada) | coco | 20 sin early stopping | `RCNN_ablacion` |
| anclas | paneles | 20 sin early stopping | `RCNN_anclas` |
| aumento | paneles + aumento fuerte | 20 sin early stopping | `RCNN_aumento` |

Con lo que se corre por defecto (anclas y aumento), *anclas − base* es el efecto conjunto de las anclas y las 20 épocas, y *aumento − anclas* el del aumento. Son 3 entrenamientos por variante, unos 30 minutos cada una en una T4: **alrededor de una hora en total**. Si la sesión se corta, al volver a correr sigue desde donde quedó.

La ablación separa el efecto de las anclas del de las 20 épocas, pero suma otra media hora: está apagada (`CORRER_ABLACION = False`). Sin ella, en el informe hay que aclarar que anclas y épocas se midieron juntas.

Los resultados van a `MyDrive/paneles_resultados/`, separados por variante y a salvo de que se corte la sesión.

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Fuera de Colab: no se montó Drive.")

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ana-castillo-zuain/procesamiento-de-imagenes.git"

# Poner en False para saltear una variante. La de anclas se corre siempre.
CORRER_AUMENTO = True
CORRER_ABLACION = False  # opcional: separa anclas de épocas, +30 min

if IN_COLAB:
    # El repo se clona en /content: son unos segundos y no hace falta subir nada a Drive.
    # /content se borra al cortarse la sesión; por eso los resultados van a Drive.
    REPO_DIR = Path("/content/procesamiento-de-imagenes")
    if REPO_DIR.is_dir():
        # fetch + reset en vez de pull: un clon --depth 1 no siempre puede hacer
        # fast-forward, y acá no hay nada local que valga la pena conservar.
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    DRIVE = Path("/content/drive/MyDrive/paneles_resultados")
    RESULTS_DIR, ABLATION_DIR, BASE_RESULTS_DIR = DRIVE / "RCNN_anclas", DRIVE / "RCNN_ablacion", DRIVE / "RCNN"
    AUGMENT_DIR = DRIVE / "RCNN_aumento"
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "RCNN" else Path.cwd()
    RCNN = REPO_DIR / "RCNN"
    RESULTS_DIR, ABLATION_DIR, BASE_RESULTS_DIR = RCNN / "results_anclas", RCNN / "results_ablacion", RCNN / "results"
    AUGMENT_DIR = RCNN / "results_aumento"

CODE_DIR = REPO_DIR / "RCNN"
DATASET_DIR = REPO_DIR / "data" / "yolo"

assert CODE_DIR.joinpath("train.py").is_file(), f"No está train.py en {CODE_DIR}"
assert REPO_DIR.joinpath("YOLO", "solar_yolo").is_dir(), "Falta YOLO/solar_yolo: solar_rcnn usa sus folds y CSV."
assert DATASET_DIR.joinpath("PV01_val").is_dir(), f"No está el dataset en {DATASET_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if CORRER_AUMENTO:
    AUGMENT_DIR.mkdir(parents=True, exist_ok=True)
if CORRER_ABLACION:
    ABLATION_DIR.mkdir(parents=True, exist_ok=True)
print("código    :", CODE_DIR)
print("dataset   :", DATASET_DIR)
print("anclas    :", RESULTS_DIR)
print("aumento   :", AUGMENT_DIR if CORRER_AUMENTO else "(no se corre)")
print("ablación  :", ABLATION_DIR if CORRER_ABLACION else "(no se corre)")

In [ ]:
import torch
import torchvision

# No se instala nada: Colab ya trae torch y torchvision con CUDA.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible. Activá una GPU en el entorno de ejecución antes de entrenar. "
        "No se cargó ningún modelo."
    )
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__, "torchvision", torchvision.__version__, "cuda", torch.version.cuda)

In [ ]:
import sys
sys.path.insert(0, str(CODE_DIR))

from solar_rcnn.config import ANCHOR_PRESETS, AUGMENT_PRESETS, SHORT_SCHEDULE, Config
from solar_rcnn.engine import verify_setup

ARQUITECTURAS = ["fasterrcnn_resnet50_fpn"]
ESTRATEGIAS = ["partial"]  # la mejor de la base; "full" empataba con ella
FOLDS = None  # None = los tres; o [1, 2, 3]

assert "paneles" in ANCHOR_PRESETS and "fuerte" in AUGMENT_PRESETS, (
    "El código clonado no tiene los presets de anclas o de aumento: falta pushear los cambios de solar_rcnn."
)


def armar_config(results_dir, anchors, augment="volteos"):
    return Config(
        dataset_dir=DATASET_DIR,
        results_dir=results_dir,
        batch=8,
        anchors=anchors,
        augment=augment,
        epochs=SHORT_SCHEDULE["epochs"],
        patience=SHORT_SCHEDULE["patience"],
    )


config_anclas = armar_config(RESULTS_DIR, "paneles")
config_aumento = armar_config(AUGMENT_DIR, "paneles", "fuerte")
config_ablacion = armar_config(ABLATION_DIR, "coco")

# verify_setup no carga modelos. Corta si la carpeta ya tiene resultados de otra configuración.
print(verify_setup(config_anclas, ARQUITECTURAS, ESTRATEGIAS, FOLDS))
if CORRER_AUMENTO:
    verify_setup(config_aumento, ARQUITECTURAS, ESTRATEGIAS, FOLDS)
if CORRER_ABLACION:
    verify_setup(config_ablacion, ARQUITECTURAS, ESTRATEGIAS, FOLDS)
print("\nLas configuraciones elegidas pasaron la verificación.")

## Entrenar

Un experimento ya terminado, con su `best.pt`, no se repite. Uno cortado a la mitad sigue desde su última época. Si la carpeta tiene resultados de **otra** configuración, corta con un error en vez de mezclarlos.

In [ ]:
from solar_rcnn.engine import run_experiments

outcome = run_experiments(config_anclas, architectures=ARQUITECTURAS, strategies=ESTRATEGIAS, folds=FOLDS)
print("fallos:", len(outcome["failures"]))
for failure in outcome["failures"]:
    print(failure["experiment_id"], failure["error"])

## Anclas + aumento fuerte

Las mismas anclas y las mismas 20 épocas, con rotaciones y cambios de color además de los volteos. Otra media hora de T4.

In [ ]:
if CORRER_AUMENTO:
    outcome_aumento = run_experiments(
        config_aumento, architectures=ARQUITECTURAS, strategies=ESTRATEGIAS, folds=FOLDS
    )
    print("fallos:", len(outcome_aumento["failures"]))
    for failure in outcome_aumento["failures"]:
        print(failure["experiment_id"], failure["error"])
else:
    print("CORRER_AUMENTO = False: no se corre el aumento.")

## Control: las mismas 20 épocas con las anclas de COCO

Es lo que permite separar el efecto de las anclas del efecto del schedule. Otra media hora de T4.

In [ ]:
if CORRER_ABLACION:
    outcome_ablacion = run_experiments(
        config_ablacion, architectures=ARQUITECTURAS, strategies=ESTRATEGIAS, folds=FOLDS
    )
    print("fallos:", len(outcome_ablacion["failures"]))
    for failure in outcome_ablacion["failures"]:
        print(failure["experiment_id"], failure["error"])
else:
    print("CORRER_ABLACION = False: no se corre el control.")

## Comparación

In [ ]:
import pandas as pd

COLUMNS = ["architecture", "strategy", "fold1_map50_95", "fold2_map50_95", "fold3_map50_95", "macro_map50_95"]
BASE, ABLACION, ANCLAS = "base (coco, 50 ép.)", "ablación (coco, 20 ép.)", "anclas (paneles, 20 ép.)"
AUMENTO = "aumento (paneles + fuerte, 20 ép.)"
VARIANTES = [(BASE, BASE_RESULTS_DIR), (ABLACION, ABLATION_DIR), (ANCLAS, RESULTS_DIR), (AUMENTO, AUGMENT_DIR)]

tablas = []
for nombre, carpeta in VARIANTES:
    archivo = carpeta / "summary.csv"
    if not archivo.is_file():
        print("falta", archivo)
        continue
    tabla = pd.read_csv(archivo)
    tabla = tabla[tabla["architecture"].isin(ARQUITECTURAS) & tabla["strategy"].isin(ESTRATEGIAS)]
    tablas.append(tabla[COLUMNS].assign(variante=nombre))

comparacion = pd.concat(tablas).sort_values(["strategy", "variante"])
display(comparacion.round(4))

In [ ]:
# Atribución: cuánto aporta cada cambio al mAP50-95 macro.
macro = comparacion.pivot_table(index="strategy", columns="variante", values="macro_map50_95")

if {BASE, ABLACION, ANCLAS} <= set(macro.columns):
    macro["efecto schedule"] = macro[ABLACION] - macro[BASE]
    macro["efecto anclas"] = macro[ANCLAS] - macro[ABLACION]
    if AUMENTO in macro.columns:
        macro["efecto aumento"] = macro[AUMENTO] - macro[ANCLAS]
    macro["total"] = macro[AUMENTO if AUMENTO in macro.columns else ANCLAS] - macro[BASE]
    display(macro.round(4))
    print(
        "Recordá el ruido: en la base, el mAP de validación se movía de 0.02 a 0.05 entre épocas "
        "consecutivas, y hay una sola semilla. Diferencias por debajo de ~0.03 no son concluyentes."
    )
elif {BASE, ANCLAS} <= set(macro.columns):
    display((macro[ANCLAS] - macro[BASE]).round(4).rename("anclas - base"))
    if AUMENTO in macro.columns:
        display((macro[AUMENTO] - macro[ANCLAS]).round(4).rename("aumento - anclas"))
    print("Sin la ablación, esta diferencia mezcla el cambio de anclas con el de schedule.")
else:
    print("Faltan variantes para comparar.")

---

Al terminar: **Archivo → Descargar → .ipynb** guarda el notebook con las salidas, y los CSV quedan en Drive. Para volcar los números al informe, `RESULTADOS.md` se arma desde `summary.csv`, `fold_metrics.csv` y `classification_metrics.csv` de cada carpeta.